# Nomadic LLM Experiment — NB04: Analysis & PAPER Tables

NB02 + NB03에서 저장된 CSV들을 통합 분석하여:
1. 비교 테이블 생성 (PAPER.md §4.5용)
2. ΔH 분해: 각 컴포넌트의 기여분 정량화
3. 시나리오별 시각화
4. 한계 자동 탐지 및 플래그

In [ ]:
# ============================================================
# STEP 0: 로드
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

DRIVE_BASE = '/content/drive/MyDrive/nomadic_llm_results'
with open(os.path.join(DRIVE_BASE, 'latest_run_dir.txt')) as f:
    lines = f.read().strip().split('\n')
    RUN_DIR, MODEL_KEY = lines[0], lines[1]

# saturation check 결과 로드
with open(os.path.join(RUN_DIR, 'saturation_check.json')) as f:
    saturation = json.load(f)

SATURATION_WARNING = saturation['sp_diff'] < 0.1

print(f'RUN_DIR : {RUN_DIR}')
print(f'MODEL   : {MODEL_KEY}')
if SATURATION_WARNING:
    print('⚠️  PolicyNet saturation 감지됨. 결과 해석 시 주의.')

In [ ]:
# ============================================================
# STEP 1: 시나리오 A — 통합 비교 테이블
# ============================================================

# 모든 모델의 시나리오 A 결과 통합
dfs = []
for fname in ['baseline_scenario_A.csv', 'nomadic_scenario_A.csv']:
    fpath = os.path.join(RUN_DIR, fname)
    if os.path.exists(fpath):
        dfs.append(pd.read_csv(fpath))
    else:
        print(f'⚠️  파일 없음: {fname}')

if not dfs:
    raise FileNotFoundError('시나리오 A CSV가 없습니다. NB02, NB03을 먼저 실행하세요.')

df_all = pd.concat(dfs, ignore_index=True)

MODEL_ORDER = ['Vanilla', 'DynamicTemp', 'StaticLoRA', 'FixedRouting',
               'Nomadic_NoDx', 'Nomadic_Full']

summary = df_all.groupby(['model', 'phase']).agg(
    mean_H   =('mean_entropy', 'mean'),
    std_H    =('std_entropy',  'mean'),
    mean_ppl =('perplexity',   'mean'),
    mean_rep =('repeat_rate',  'mean'),
    mean_sw  =('switch_rate',  'mean'),
).round(4).reset_index()

pivot = summary.pivot_table(
    index='model', columns='phase', values='mean_H'
).reset_index()
pivot['ΔH'] = pivot.get('transition', 0) - pivot.get('stable', 0)

# PPL (language domain만)
ppl_lang = df_all[df_all['domain']=='language'].groupby('model')['perplexity'].mean().reset_index()
ppl_lang.columns = ['model', 'lang_ppl']
pivot = pivot.merge(ppl_lang, on='model', how='left')

# RepeatRate (language domain만)
rep_lang = df_all[df_all['domain']=='language'].groupby('model')['repeat_rate'].mean().reset_index()
rep_lang.columns = ['model', 'lang_rep']
pivot = pivot.merge(rep_lang, on='model', how='left')

print('='*90)
print(f'§4.5 LLM Experiment — {MODEL_KEY} (3-Expert)')
print('='*90)
print('\n--- Table: ΔH + PPL + RepeatRate ---')
print(f'{"Model":20s} | Stable H | Trans H |   ΔH  | Lang PPL | RepRate')
print('-'*75)
for m in MODEL_ORDER:
    row = pivot[pivot['model']==m]
    if len(row) == 0: continue
    sh   = row['stable'].values[0]     if 'stable' in row.columns     else float('nan')
    th   = row['transition'].values[0] if 'transition' in row.columns else float('nan')
    dh   = row['ΔH'].values[0]
    ppl  = row['lang_ppl'].values[0]
    rep  = row['lang_rep'].values[0]
    print(f'{m:20s} | {sh:.3f}    | {th:.3f}   | {dh:+.3f} | {ppl:.3f}    | {rep:.3f}')

# PAPER.md용 마크다운
print('\n--- PAPER.md 반영용 Markdown ---')
print('| Model | Stable H | Trans H | ΔH | Lang PPL | Rep Rate |')
print('|-------|----------|---------|-----|----------|----------|')
for m in MODEL_ORDER:
    row = pivot[pivot['model']==m]
    if len(row) == 0: continue
    sh  = row['stable'].values[0]     if 'stable'     in row.columns else float('nan')
    th  = row['transition'].values[0] if 'transition' in row.columns else float('nan')
    dh  = row['ΔH'].values[0]
    ppl = row['lang_ppl'].values[0]
    rep = row['lang_rep'].values[0]
    print(f'| {m} | {sh:.3f} | {th:.3f} | {dh:+.3f} | {ppl:.3f} | {rep:.3f} |')

pivot.to_csv(os.path.join(RUN_DIR, 'table_scenario_A.csv'), index=False)

In [ ]:
# ============================================================
# STEP 2: 컴포넌트 기여분 분해 테이블
# ============================================================
# 각 컴포넌트 추가 시 ΔH 변화:
#   Vanilla → DynamicTemp    : Δx 신호 기여
#   Vanilla → StaticLoRA     : LoRA expert 기여
#   StaticLoRA → FixedRouting: 도메인 라우팅 기여
#   FixedRouting → Nomadic_NoDx: PolicyNet 기여 (Δx 없이)
#   Nomadic_NoDx → Nomadic_Full: Δx 신호 기여 (PolicyNet과 결합)

print('\n--- 컴포넌트 기여분 분해 ---')
dh_map = {}
for m in MODEL_ORDER:
    row = pivot[pivot['model']==m]
    if len(row): dh_map[m] = float(row['ΔH'].values[0])

steps = [
    ('Baseline (Vanilla)',            'Vanilla',       None,            dh_map.get('Vanilla', float('nan'))),
    ('+Δx signal (DynamicTemp)',      'DynamicTemp',   'Vanilla',       dh_map.get('DynamicTemp', float('nan'))),
    ('+LoRA expert (StaticLoRA)',     'StaticLoRA',    'Vanilla',       dh_map.get('StaticLoRA', float('nan'))),
    ('+Domain routing (FixedRouting)','FixedRouting',  'StaticLoRA',    dh_map.get('FixedRouting', float('nan'))),
    ('+PolicyNet (NoDx)',             'Nomadic_NoDx',  'FixedRouting',  dh_map.get('Nomadic_NoDx', float('nan'))),
    ('+Δx (Full)',                    'Nomadic_Full',  'Nomadic_NoDx',  dh_map.get('Nomadic_Full', float('nan'))),
]

print(f'{"Component":40s} | ΔH    | Gain vs prev')
print('-'*65)
prev_dh = None
for label, model, prev_model, dh in steps:
    gain = (dh - dh_map.get(prev_model, dh)) if prev_model else 0.0
    if prev_model is None:
        print(f'{label:40s} | {dh:+.4f} | —')
    else:
        arrow = '↑' if gain > 0 else '↓'
        print(f'{label:40s} | {dh:+.4f} | {arrow} {abs(gain):.4f}')

print('\n--- PAPER.md 반영용 ---')
print('| Component | ΔH | Gain vs prev |')
print('|-----------|-----|------|')
for label, model, prev_model, dh in steps:
    gain_str = '—' if not prev_model else f'{dh - dh_map.get(prev_model, dh):+.4f}'
    print(f'| {label} | {dh:+.4f} | {gain_str} |')

In [ ]:
# ============================================================
# STEP 3: 시나리오 B~G 통합 집계
# ============================================================

def load_scenario(scenario, models=None):
    """baseline + nomadic CSV 통합 로드"""
    dfs = []
    for prefix in ['baseline', 'nomadic']:
        fpath = os.path.join(RUN_DIR, f'{prefix}_scenario_{scenario}.csv')
        if os.path.exists(fpath):
            dfs.append(pd.read_csv(fpath))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


# ── 시나리오 E: 유혹-회복 ─────────────────────────────────
df_E = load_scenario('E')
if not df_E.empty:
    e_summary = df_E.groupby('model').agg(
        mean_lure_spike    =('lure_spike',      'mean'),
        mean_recovery_steps=('recovery_steps',  'mean'),
    ).round(3).reset_index()
    print('\n--- 시나리오 E: 유혹-회복 ---')
    print(e_summary.to_string(index=False))


# ── 시나리오 F: 정보 밀도 ─────────────────────────────────
df_F = load_scenario('F')
if not df_F.empty:
    print('\n--- 시나리오 F: 정보 밀도 ---')
    print(df_F[['model','density','mean_dx','corr_dx_conf']].to_string(index=False))


# ── 시나리오 G: 장기 고착도 ───────────────────────────────
df_G = load_scenario('G')
if not df_G.empty:
    g_summary = df_G.groupby('model').agg(
        mean_conv =('convergence_step','mean'),
        mean_decay=('decay_rate',      'mean'),
    ).round(3).reset_index()
    print('\n--- 시나리오 G: 장기 고착도 ---')
    print(g_summary.to_string(index=False))


# ── 시나리오 D (Nomadic_Full only) ────────────────────────
fpath_D = os.path.join(RUN_DIR, 'nomadic_scenario_D.csv')
if os.path.exists(fpath_D):
    df_D = pd.read_csv(fpath_D)
    print('\n--- 시나리오 D: 도메인 중첩 (Nomadic_Full) ---')
    print(df_D[['prompt','oscillation','dominant_expert','switch_rate']].to_string(index=False))

In [ ]:
# ============================================================
# STEP 4: 시각화
# ============================================================

COLORS = {
    'Vanilla':      '#888888',
    'DynamicTemp':  '#E07B54',
    'StaticLoRA':   '#4CAF50',
    'FixedRouting': '#9C27B0',
    'Nomadic_NoDx': '#FF9800',
    'Nomadic_Full': '#2C6FAC',
}

models_present = [m for m in MODEL_ORDER if m in pivot['model'].values]

# ── Figure 1: ΔH 분해 (메인 결과) ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'§4.5 LLM Experiment — {MODEL_KEY} (3-Expert)', fontsize=12, fontweight='bold')

# ΔH bar
ax = axes[0]
dh_vals = [float(pivot[pivot['model']==m]['ΔH'].values[0])
            if m in pivot['model'].values else float('nan') for m in models_present]
bars = ax.bar(models_present, dh_vals,
              color=[COLORS.get(m,'gray') for m in models_present], width=0.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('ΔH = Trans H − Stable H\n(Higher = Better Context Differentiation)', fontsize=9)
ax.set_ylabel('ΔH'); ax.tick_params(axis='x', rotation=30, labelsize=8)
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, dh_vals):
    if not np.isnan(v):
        ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.003,
                f'{v:+.3f}', ha='center', fontsize=7, fontweight='bold')

# Language PPL
ax = axes[1]
ppl_vals = [float(pivot[pivot['model']==m]['lang_ppl'].values[0])
             if m in pivot['model'].values else float('nan') for m in models_present]
bars = ax.bar(models_present, ppl_vals,
              color=[COLORS.get(m,'gray') for m in models_present], width=0.5)
ax.set_title('Language Domain PPL\n(Lower = Better Generation Quality)', fontsize=9)
ax.set_ylabel('PPL'); ax.tick_params(axis='x', rotation=30, labelsize=8)
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, ppl_vals):
    if not np.isnan(v):
        ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.01,
                f'{v:.3f}', ha='center', fontsize=7)

# Repeat Rate
ax = axes[2]
rep_vals = [float(pivot[pivot['model']==m]['lang_rep'].values[0])
             if m in pivot['model'].values else float('nan') for m in models_present]
bars = ax.bar(models_present, rep_vals,
              color=[COLORS.get(m,'gray') for m in models_present], width=0.5)
ax.set_title('Language Domain Repeat Rate\n(Lower = Less Degeneration)', fontsize=9)
ax.set_ylabel('Repeat Rate'); ax.tick_params(axis='x', rotation=30, labelsize=8)
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, rep_vals):
    if not np.isnan(v):
        ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.002,
                f'{v:.3f}', ha='center', fontsize=7)

if SATURATION_WARNING:
    fig.text(0.5, 0.01, '⚠️ PolicyNet saturation detected — interpret ΔH with caution',
             ha='center', fontsize=8, color='red')

plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, 'fig1_main_comparison.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('fig1_main_comparison.png 저장')


# ── Figure 2: 도메인별 entropy 분포 ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Entropy by Domain — {MODEL_KEY}', fontsize=12, fontweight='bold')

domains = ['math', 'code', 'language']
x = np.arange(len(models_present))
bw = 0.13

for col, domain in enumerate(domains):
    ax = axes[col]
    for i, m in enumerate(models_present):
        sub = df_all[(df_all['model']==m) & (df_all['domain']==domain)]
        if sub.empty: continue
        val = sub['mean_entropy'].mean()
        err = sub['mean_entropy'].std()
        ax.bar(i, val, 0.6, yerr=err, capsize=3,
               color=COLORS.get(m,'gray'), alpha=0.8,
               label=m if col==0 else None)
    ax.set_title(f'{domain.capitalize()} Domain', fontsize=10)
    ax.set_xticks(range(len(models_present)))
    ax.set_xticklabels(models_present, rotation=30, fontsize=7)
    ax.set_ylabel('Entropy'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, 'fig2_entropy_by_domain.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('fig2_entropy_by_domain.png 저장')


# ── Figure 3: Scenario B Δx trace ──────────────────────────
trace_path = os.path.join(RUN_DIR, 'scenario_B_dx_trace.png')
if os.path.exists(trace_path):
    print('scenario_B_dx_trace.png 이미 존재 (NB02에서 생성됨)')

print('\n✅ NB04 완료')

In [ ]:
# ============================================================
# STEP 5: 한계 자동 탐지 및 플래그
# ============================================================
print('='*60)
print('한계 탐지 리포트')
print('='*60)

flags = []

# 1. PolicyNet saturation
if SATURATION_WARNING:
    flags.append('[WARN] PolicyNet switch saturation: math/language 간 '
                 f'switch_prob 차이 = {saturation["sp_diff"]:.3f} (< 0.1). '
                 'ΔH 차이가 나타나더라도 PolicyNet 기여가 아닐 수 있음.')

# 2. Nomadic_Full vs DynamicTemp ΔH 차이 (작으면 LoRA 기여가 없는 것)
full_dh = dh_map.get('Nomadic_Full', float('nan'))
dt_dh   = dh_map.get('DynamicTemp', float('nan'))
if not (np.isnan(full_dh) or np.isnan(dt_dh)):
    if (full_dh - dt_dh) < 0.05:
        flags.append(f'[WARN] Nomadic_Full ΔH ({full_dh:.3f}) vs DynamicTemp ΔH ({dt_dh:.3f}) '
                     '차이가 작음 (< 0.05). LoRA+PolicyNet의 추가 기여가 미미할 수 있음.')

# 3. FixedRouting vs Nomadic_Full 비교
fr_dh = dh_map.get('FixedRouting', float('nan'))
if not (np.isnan(full_dh) or np.isnan(fr_dh)):
    if full_dh < fr_dh:
        flags.append(f'[WARN] Nomadic_Full ΔH ({full_dh:.3f}) < FixedRouting ΔH ({fr_dh:.3f}). '
                     '동적 라우팅이 정적 라우팅보다 나쁜 결과. '
                     'PolicyNet 학습 실패 또는 도메인 라우팅이 이미 최적일 가능성.')

# 4. Entropy std 체크 (재현성)
high_std = df_all.groupby('model')['std_entropy'].mean()
for m, s in high_std.items():
    if s > 0.15:
        flags.append(f'[WARN] {m}: mean std_entropy = {s:.3f} (> 0.15). '
                     f'결과 재현성이 낮음. N_RUNS 증가 권장.')

if not flags:
    print('✅ 자동 탐지된 한계 없음')
else:
    for flag in flags:
        print(flag)
        print()

# 저장
with open(os.path.join(RUN_DIR, 'limitation_flags.txt'), 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(flags) if flags else 'No limitations detected.')

print('\n' + '='*60)
print('모든 결과 파일 저장 위치:')
print(f'  {RUN_DIR}')
for fname in sorted(os.listdir(RUN_DIR)):
    print(f'  {fname}')